---
downloads:
  - file: tp05_chute_avec_frottement_teacher.ipynb
    title: Notebook enseignant
    filename: tp05_chute_avec_frottement_enseignant.ipynb
  - file: ../notebooks_student/tp05_chute_avec_frottement_student.ipynb
    title: Notebook étudiant
    filename: tp05_chute_avec_frottement_etudiant.ipynb
---

# TP 05 — Chute avec frottement : résolution numérique

Dans ce TP, nous étudions la chute d'un point matériel soumis à des
frottements de l'air, en résolvant l'équation différentielle du
mouvement de façon analytique puis numérique.

:::{important} Version enseignant
Cette page présente la version **corrigée**, destinée aux enseignants.

Pour télécharger la version à distribuer aux étudiants et à déposer sur
Boostcamp, utilisez le bouton **Notebook étudiant** dans le menu de
téléchargement situé en haut de la page.

Le lien suivant ouvre la version enseignant dans Google Colab :
[Ouvrir dans Google Colab](https://colab.research.google.com/github/ECE-Ecole-Ingenieurs/cours-physique/blob/main/notebooks/tp05_chute_avec_frottement_teacher.ipynb)
:::

**Importation bibliothèques**

In [ ]:
import numpy as np  # Manipulation des vecteurs et opérations matricielles
import matplotlib.pyplot as plt  # Tracé des figures 2D et 3D
from ipywidgets import interact, FloatSlider, IntSlider  # Widgets interactifs (curseurs)

## 1. Rappel de cours

Comme au TP précédent, nous partons du principe fondamental de la
dynamique :

$$
\overrightarrow{F} = m\overrightarrow{a}(M)_{/R}
$$

Lorsque les frottements de l'air interviennent, la force de frottement
dépend de la vitesse du point matériel : le PFD projeté sur un axe
conduit alors à une équation différentielle portant sur la vitesse, et
non plus directement à une expression explicite de l'accélération.

Cela revient à résoudre une équation différentielle linéaire du premier
ordre de la forme :

$$
y'(t) + \frac{1}{\tau}y(t) = A
$$

avec $A$ et $\tau$ des constantes. L'ensemble des solutions de cette
équation est de la forme $y=y_p+y_h$, où :

- $y_h$ est solution de l'équation homogène $y'(t)+\frac1\tau y(t)=0$,
  soit $y_h(t)=Ke^{-t/\tau}$ ;
- $y_p$ est une solution particulière constante de l'équation complète,
  soit $y_p=A\tau$.

La solution générale s'écrit donc :

$$
y(t) = A\tau + Ke^{-t/\tau}
$$

où $K$ est déterminée par la condition initiale.

Certaines équations différentielles admettent une solution analytique
exacte, comme dans le cas d'un frottement linéaire. Pour d'autres,
comme le frottement quadratique, la solution n'a pas toujours une
forme simple : on utilise alors une méthode numérique.

Vous avez vu en mathématiques la méthode d'Euler explicite, qui permet
d'approximer la solution d'une équation différentielle du premier
ordre.

On considère une équation sous la forme

$$
y'(t) = F(t, y(t)),
$$

avec, dans notre cas,

$$
F(t,y) = A - \frac{1}{\tau}y.
$$

On subdivise la durée de l'étude en $n$ pas réguliers, c'est-à-dire en
une liste de $n+1$ instants successifs :

$$
t_0,\ t_1,\ t_2,\ \ldots,\ t_i,\ t_{i+1},\ \ldots,\ t_n
$$

où $t_i$ désigne le $i$-ème instant de la liste et $t_{i+1}$ l'instant
suivant. Le pas $h$ est la durée constante entre deux instants
successifs :

$$
h = t_{i+1} - t_i = \frac{t_n - t_0}{n}.
$$

La valeur $y_0=y(t_0)$ à l'instant initial est supposée connue : c'est
la condition initiale.

À partir d'une valeur connue $y_i=y(t_i)$, la méthode d'Euler permet
d'estimer la valeur au pas suivant :

$$
y_{i+1} = y_i + h\,F(t_i, y_i).
$$

On calcule ainsi la solution pas à pas, de $t_0$ jusqu'à $t_n$.

## 2. Chute avec frottement linéaire

On reprend la chute étudiée au TP précédent (axe $(Oz)$ vertical
ascendant, point $M$ de masse $m$ lâché sans vitesse initiale d'une
hauteur $h_0$), en tenant compte cette fois des frottements de l'air.
On modélise la force de frottement par
$\overrightarrow{f}=-\lambda\overrightarrow{v}$, où $\lambda$ est un
coefficient positif.

### 2.1 Théorie

:::{admonition} ✍️ Questions
:class: tip

1. Déterminer l'équation différentielle d'ordre 1 régissant la
   composante $v_z(t)$ de la vitesse du point $M$.
2. En déduire l'expression de $v_z(t)$.
:::

:::{admonition} 💡 Aide
:class: hint

Projetez le PFD sur l'axe $(Oz)$, puis identifiez $\tau$ et $A$ dans
l'équation obtenue pour appliquer la méthode de résolution rappelée
ci-dessus.
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --

:::{admonition} Réponses
:class: note

1. Le PFD projeté sur $(Oz)$ donne $-mg-\lambda v_z = m\dot v_z$,
   soit :

   $$
   \dot v_z + \frac{\lambda}{m}v_z = -g
   $$
2. C'est une équation de la forme $y'(t)+\frac1\tau y(t)=A$ avec
   $y=v_z$, $\tau=m/\lambda$ et $A=-g$. Avec $v_z(0)=0$, on trouve
   $K=g\tau$, d'où :

   $$
   v_z(t) = g\tau\left(e^{-t/\tau}-1\right)
   $$
:::

:::{admonition} Appelez le professeur
:class: warning

Avant de passer à l'implémentation, faites valider votre raisonnement par
l'enseignant.
:::

### 2.2 Implémentation numérique

:::{admonition} 💻 Code
:class: tip

Complétez deux fonctions :

1. `vitesse_exacte_frottement_lineaire`, qui renvoie la solution exacte
   $v_z(t)$ trouvée à la section précédente ;
2. `euler_vitesse_lineaire`, qui intègre $v_z(t)$ par la méthode
   d'Euler, à partir de l'équation différentielle établie précédemment
   et de la relation de récurrence $v_{i+1}=v_i+hF(t_i,v_i)$.
:::

In [ ]:
def vitesse_exacte_frottement_lineaire(t, m, lam):
    """
    Calcule la vitesse exacte v_z(t) lors d'une chute avec frottement
    linéaire.

    Paramètres
    ----------
    t : float ou np.ndarray
        Instant(s) considéré(s) [s].
    m : float
        Masse du point matériel [kg].
    lam : float
        Coefficient de frottement linéaire [kg/s].

    Renvoie
    -------
    v : float ou np.ndarray
        Vitesse v_z(t) [m/s].
    """
    g = 9.81
    tau = m / lam
    v = g * tau * (np.exp(-t / tau) - 1.0)

    return v


def euler_vitesse_lineaire(t_vals, m, lam):
    """
    Intègre par la méthode d'Euler explicite la vitesse v_z(t) lors
    d'une chute avec frottement linéaire.

    Paramètres
    ----------
    t_vals : np.ndarray
        Instants successifs, régulièrement espacés [s]. On suppose
        v_z(t_vals[0]) = 0.
    m : float
        Masse du point matériel [kg].
    lam : float
        Coefficient de frottement linéaire [kg/s].

    Renvoie
    -------
    v_vals : np.ndarray
        Vitesse v_z approchée aux instants t_vals [m/s].
    """
    g = 9.81
    h = t_vals[1] - t_vals[0]
    v_vals = np.zeros_like(t_vals)

    for i in range(len(t_vals) - 1):
        v_vals[i + 1] = v_vals[i] + h * (-g - (lam / m) * v_vals[i])

    return v_vals

### 2.3 Analyse

Le code suivant compare la vitesse exacte, la vitesse approchée par la
méthode d'Euler, et la vitesse sans frottement (rappel du TP
précédent). Exécutez-le, puis faites varier $\lambda$ et le nombre de
points $n_t$ utilisés pour la méthode d'Euler.

In [ ]:
def tracer_vitesse_lineaire(m=1.0, lam=0.5, n_t=100):
    """Compare la vitesse exacte, la vitesse d'Euler et la vitesse sans frottement."""
    g = 9.81
    t_max = 15.0

    t_fin = np.linspace(0.0, t_max, 300)
    v_exacte = vitesse_exacte_frottement_lineaire(t_fin, m, lam)
    v_sans_frottement = -g * t_fin

    t_euler = np.linspace(0.0, t_max, n_t)
    v_euler = euler_vitesse_lineaire(t_euler, m, lam)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(t_fin, v_sans_frottement, color='0.6', linestyle='--', linewidth=2, label='sans frottement')
    ax.plot(t_fin, v_exacte, color='tab:blue', linewidth=2, label='solution exacte')
    ax.plot(t_euler, v_euler, color='tab:red', marker='o', markersize=3, linewidth=1, label=f'Euler ({n_t} points)')
    ax.set_ylim(-160, 20)
    ax.set_xlabel('t (s)')
    ax.set_ylabel('v_z (m/s)')
    ax.set_title('Chute avec frottement linéaire')
    ax.grid(alpha=0.3)
    ax.legend(loc='lower left')

    plt.show()
    plt.close(fig)

tracer_vitesse_lineaire()

In [ ]:
interact(
    tracer_vitesse_lineaire,
    m=FloatSlider(value=1.0, min=0.5, max=3.0, step=0.5, description='m (kg)'),
    lam=FloatSlider(value=0.5, min=0.1, max=5.0, step=0.1, description='λ (kg/s)'),
    n_t=IntSlider(value=100, min=5, max=200, step=5, description='n_t'),
);

:::{admonition} 🔍 Exploration
:class: tip

1. Comparez la vitesse exacte et la vitesse approchée par la méthode
   d'Euler. Que se passe-t-il lorsque vous diminuez fortement $n_t$
   (donc lorsque le pas $h$ augmente) ?
2. Faites varier $\lambda$. Quelle est son influence sur la vitesse
   limite atteinte par le point $M$ ?
3. Comparez la vitesse avec et sans frottement. Que devient la vitesse
   sans frottement lorsque $t$ est grand ? Est-ce physique ?
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --
3. --

:::{admonition} Réponses
:class: note

1. Pour $n_t$ petit (pas $h$ grand), la méthode d'Euler s'écarte
   fortement de la solution exacte, jusqu'à diverger complètement pour
   les valeurs les plus faibles de $n_t$. Pour $n_t$ grand (pas $h$
   petit), les deux courbes deviennent quasiment confondues.
2. Plus $\lambda$ est grand, plus la vitesse limite
   $|v_z|=g\tau=gm/\lambda$ est petite : le frottement freine
   d'autant plus la chute.
3. Sans frottement, $v_z(t)=-gt$ croît indéfiniment en valeur absolue,
   ce qui n'est pas physique : l'air finit toujours par ralentir la
   chute. Avec frottement, la vitesse tend vers une valeur limite
   finie.
:::

:::{important}
Pour afficher la version interactive, ouvrez le **Notebook enseignant** en
local ou ouvrez ce notebook dans Google Colab.
:::